# KITTI Tracklet Classification — 3D ResNet + Temporal FPN + BiLSTM

A thin driver over the modules in `src/`. Every function called here is defined
and tested in the package, so this notebook stays reproducible.

**Pipeline:** parse tracklet XML → project each 3D box into the image →
crop the object → cache clips → 3D ResNet + geometry branch → temporal head.

In [ ]:
# Run from anywhere inside the repo: put the project root on sys.path.
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
from src.config import CLASS_NAMES, CANONICAL_SPLIT, TARGET_FRAMES, VIDEO_NAMES
from src.utils import get_device, set_seed

set_seed()
device = get_device()
print("device:", device)
print("classes:", CLASS_NAMES)
print("window:", TARGET_FRAMES, "frames")

## 1. Dataset composition

Measured from the XML, not assumed.

In [ ]:
import collections
from src.dataset import parse_tracklets

tracklets = [t for v in VIDEO_NAMES for t in parse_tracklets(v)]
counts = collections.Counter(t.label for t in tracklets)
lengths = [t.length for t in tracklets]

print(f"tracklets: {len(tracklets)}")
print(f"tracklet-frames: {sum(lengths)}")
print(f"shorter than {TARGET_FRAMES} frames: {sum(1 for n in lengths if n < TARGET_FRAMES)}")
print(f"start after frame 0: {sum(1 for t in tracklets if t.first_frame > 0)}")
for name, n in counts.most_common():
    print(f"  {name:<12}{n:>4}")

## 2. Build the crop cache

Each tracklet box is projected into its frame with the calibration matrix and
cropped. This is the step that makes the visual branch object-specific: without
it, every tracklet in a video receives an identical whole-frame descriptor.

Run once; skip if `data/crop_cache/` is already populated.

In [ ]:
from src.dataset import build_crop_cache

# stats = build_crop_cache(overwrite=True)   # ~100 s for all nine sequences
# print({k: v for k, v in stats.items() if k != "per_video"})

### Regression check: descriptors must be unique per tracklet

In [ ]:
import numpy as np
from src.dataset import index_cache

entries = index_cache(["Video_13"])
signatures = []
for entry in entries:
    with np.load(entry.path, allow_pickle=True) as data:
        crops, valid = data["crops"], data["valid"]
    signatures.append(crops[valid][0].astype(float).mean(axis=(0, 1)))

unique = len(np.unique(np.round(np.array(signatures), 4), axis=0))
print(f"unique first-crop descriptors: {unique} / {len(entries)} tracklets")
assert unique == len(entries), "crops are not object-specific"

## 3. Inspect a cached clip

In [ ]:
import matplotlib.pyplot as plt

with np.load(entries[0].path, allow_pickle=True) as data:
    crops, valid, label = data["crops"], data["valid"], str(data["label_name"])

frames = np.flatnonzero(valid)[:8]
fig, axes = plt.subplots(1, len(frames), figsize=(16, 2.6))
for ax, f in zip(axes, frames):
    ax.imshow(crops[f])
    ax.set_title(f"t={f}", fontsize=9)
    ax.axis("off")
fig.suptitle(f"Cached object crops — {label}")
plt.tight_layout()
plt.show()

## 4. Model

In [ ]:
import torch
from src.model_3dresnet import TrackletClassifier
from src.utils import count_parameters

model = TrackletClassifier(head="temporal_fpn_bilstm").to(device)
trainable, total = count_parameters(model)
print(f"parameters: {trainable:,} trainable / {total:,} total")

for stage in (1, 2, 3):
    model.set_stage(stage)
    tr, tot = model.trainable_backbone_parameters()
    print(f"  stage {stage}: backbone trainable {tr:>11,} / {tot:,}")

## 5. Leave-one-video-out cross-validation

The headline protocol. A single video-level split cannot support per-class
metrics here — Video_13 has no Pedestrian and no Cyclist tracklets — so every
sequence takes a turn as the test fold and results are pooled.

These run on precomputed frozen-backbone features, so a nine-fold sweep is fast.

In [ ]:
from src.train import leave_one_video_out, precompute_embeddings
from src.dataset import index_cache

all_entries = index_cache()
records = precompute_embeddings(all_entries, device)
print(f"embeddings ready for {len(records)} tracklets")

In [ ]:
from src.evaluate import text_report

summary, probs, preds, labels = leave_one_video_out(
    records, "temporal_fpn_bilstm", device, epochs=40)

print(f"macro F1: {summary['macro_f1_mean']:.3f} +/- {summary['macro_f1_std']:.3f}")
print(f"accuracy: {summary['accuracy_mean']:.3f} +/- {summary['accuracy_std']:.3f}")
print()
print(text_report(preds, labels))

In [ ]:
from src.evaluate import plot_confusion_matrix

path = plot_confusion_matrix(preds, labels,
                             "Leave-one-video-out (pooled, n=293)",
                             "confusion_matrix_lovo.png")
print("saved", path)

## 6. Temporal head ablation

The five variants compared under an identical protocol.

In [ ]:
from src.lstm_head import HEAD_VARIANTS

results = {}
for head_name in HEAD_VARIANTS:
    results[head_name], *_ = leave_one_video_out(records, head_name, device, epochs=40)
    r = results[head_name]
    print(f"{head_name:<22} macro F1 {r['macro_f1_mean']:.3f} +/- {r['macro_f1_std']:.3f}")

In [ ]:
from src.evaluate import plot_ablation
print("saved", plot_ablation(results))

## 7. End-to-end staged fine-tuning

Unfreezes the 3D ResNet progressively with augmentation enabled. Slower than the
frozen-feature path above — run it when you want the final model rather than a
protocol comparison.

In [ ]:
# from src.train import train_end_to_end
#
# train_entries = [e for e in all_entries if e.video in CANONICAL_SPLIT["train"]]
# val_entries   = [e for e in all_entries if e.video in CANONICAL_SPLIT["val"]]
# model, histories, stages, geometry_stats = train_end_to_end(
#     train_entries, val_entries, device)